## <small>
Copyright (c) 2017-21 Andrew Glassner

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
</small>



# Deep Learning: A Visual Approach
## by Andrew Glassner, https://glassner.com
### Order: https://nostarch.com/deep-learning-visual-approach
### GitHub: https://github.com/blueberrymusic
------

### What's in this notebook

This notebook is provided as a “behind-the-scenes” look at code used to make some of the figures in this chapter. It is cleaned up a bit from the original code that I hacked together, and is only lightly commented. I wrote the code to be easy to interpret and understand, even for those who are new to Python. I tried never to be clever or even more efficient at the cost of being harder to understand. The code is in Python3, using the versions of libraries as of April 2021. 

This notebook may contain additional code to create models and images not in the book. That material is included here to demonstrate additional techniques.

Note that I've included the output cells in this saved notebook, but Jupyter doesn't save the variables or data that were used to generate them. To recreate any cell's output, evaluate all the cells from the start up to that cell. A convenient way to experiment is to first choose "Restart & Run All" from the Kernel menu, so that everything's been defined and is up to date. Then you can experiment using the variables, data, functions, and other stuff defined in this notebook.

## Chapter 19: RNNs - Notebook 5: Sunspots

In [4]:
import numpy as np
from keras.models import Sequential
from keras.layers import LSTM, Dense, LeakyReLU
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.optimizers import RMSprop
from keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
import math
import seaborn as sns ; sns.set()

random_seed = 42

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
def sum_of_mixed_sines(num_steps):
    values = [math.sin(12*t) * math.cos(t) + (math.sin(3*t) * math.sin(t/2.3)) \
              for t in np.linspace(0, 4 * np.pi, num_steps)]
    return np.array(values)

In [ ]:
def get_sunspot_data():
    filename = 'input_data/sunspots.txt'
    with open(filename) as f:
        content = f.readlines()
    data = [float(x.strip()) for x in content] 
    data = np.array(data)
    return data

In [ ]:
def make_raw_data(training_length, test_length, data_source):
    if data_source == 'sines':
        X_train = sum_of_mixed_sines(training_length)
        X_test = sum_of_mixed_sines(test_length)
    else:
        data = get_sunspot_data()
        train_len = int(len(data) * training_length / (training_length + test_length))
        train_len = 2752
        X_train = data[:train_len]
        X_test = data[train_len:]
    return (X_train, X_test)

In [ ]:
def samples_and_targets_from_sequence(sequence, window_size):
    '''Return lists of samples and targets built from overlapping
    windows of the given size. Windows start at the beginning of 
    the input sequence and move right by 1 element.'''
    samples = []
    targets = []
    for i in range(sequence.shape[0]-window_size):
        sample = sequence[i:i+window_size]
        target = sequence[i+window_size]
        samples.append(sample)
        targets.append(target[0]) 
    return (np.array(samples), np.array(targets))

In [ ]:
def scale_sequences(training_sequence, test_sequence):
    # reshape train and test sequences to form needed by MinMaxScaler
    training_sequence = np.reshape(training_sequence, (training_sequence.shape[0], 1))
    test_sequence = np.reshape(test_sequence, (test_sequence.shape[0], 1))
    min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
    min_max_scaler.fit(training_sequence)
    scaled_training_sequence = min_max_scaler.transform(training_sequence)
    scaled_test_sequence = min_max_scaler.transform(test_sequence)
    return (min_max_scaler, scaled_training_sequence, scaled_test_sequence)

In [ ]:
def chop_up_sequences(training_sequence, test_sequence, window_size):
    (X_train, y_train) = samples_and_targets_from_sequence(training_sequence, window_size)
    (X_test, y_test) = samples_and_targets_from_sequence(test_sequence, window_size)
    return (X_train, y_train, X_test, y_test)

In [ ]:
def make_data_set(window_size, training_length, data_source):
    testing_length = 0.5 * training_length
    train_seq, test_seq = make_raw_data(training_length, testing_length, data_source)
    min_max_scaler, scaled_train_seq, scaled_test_seq = scale_sequences(train_seq, test_seq)
    X_train, y_train, X_test, y_test = chop_up_sequences(scaled_train_seq, scaled_test_seq, window_size)
    return (min_max_scaler, X_train, y_train, X_test, y_test, train_seq, test_seq)

In [5]:
def plot_one_curve(data, title, linewidth=2, filename=None):
    plt.figure(figsize=(8,3))
    plt.plot(data, lw=linewidth)
    plt.title(title)
    if filename is not None:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [6]:
X_train, X_test = make_raw_data(500, 200, 'spots')
plot_one_curve(X_train, 'Raw training data', 0.5, 'spots_raw_training_data.png')

NameError: name 'make_raw_data' is not defined

In [ ]:
mmscaler, tr, te = scale_sequences(X_train, X_test)
plot_one_curve(tr, 'Normalized training data', 0.5, 'spots_normalized_training_data.png')

In [ ]:
# build and run the first model. 
def make_rnn_model(units_list, window_size):
    model = Sequential()
    for i, num_units in enumerate(units_list):
        if i == 0:
            ret_seq = len(units_list) > 1
            model.add(LSTM(num_units, return_sequences = ret_seq, input_shape=[window_size, 1]))
        else:
            ret_seq = i != len(units_list)-1
            model.add(LSTM(num_units, return_sequences = ret_seq))

    model.add(Dense(1, activation=None))

    model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])
    model.summary()
    return model

In [ ]:
def make_dense_model(units_list, window_size):
    model = Sequential()
    for i in range(len(units_list)):
        if i==0:
            model.add(Dense(units_list[i], activation='linear', input_shape=[window_size]))
        else:
            model.add(Dense(units_list[i], activation='linear'))
        model.add(LeakyReLU(alpha=0.1))   
    model.add(Dense(1))

    model.compile(loss='mse', optimizer=RMSprop(0.001), metrics=['accuracy', 'mae', 'mse'])
    model.summary()
    return model

In [ ]:
def build_and_show(model_type, data_source, units_list, window_size, training_length, 
                   epochs, linewidth=2, filename=None):
    np.random.seed(random_seed)
    min_max_scaler, X_train, y_train, X_test, y_test, train_data, test_data = \
        make_data_set(window_size, training_length, data_source)
    if model_type == 'RNN':
        model = make_rnn_model(units_list, window_size)
    else:
        model = make_dense_model(units_list, window_size)
        # data is made for RNN, so reshape
        X_train = np.reshape(X_train, X_train.shape[0:2])
        X_test = np.reshape(X_test, X_test.shape[0:2])
        
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=3, min_delta=.001)
    history = model.fit(X_train, y_train, validation_split=0.33, epochs=epochs, batch_size=1, verbose=1, callbacks=[es])
    
    # Predict 
    y_train_predict = np.ravel(model.predict(X_train))
    y_test_predict = np.ravel(model.predict(X_test))
    
    # invert transformation
    inverse_y_train_predict = min_max_scaler.inverse_transform([y_train_predict])
    inverse_y_test_predict = min_max_scaler.inverse_transform([y_test_predict])  
    
    model_string = model_type+' '+str(units_list)
    
    plt.figure(figsize=(6,6))
    plt.plot(history.history['loss'], c='#c04f54', label='training loss')
    plt.plot(history.history['val_loss'], c='#5ca66a', label='validation loss')
    plt.title('Training loss, model '+model_string)
    plt.legend(loc='best')
    if filename is not None:
        plt.savefig('loss-'+filename, dpi=300, bbox_inches='tight')
    plt.show()
       
    plt.figure(figsize=(12, 4))
    
    zfar = 10
    znear = 20
    blue_color = '#4d74ae'
    orange_color = '#f6712a'
    
    plt.subplot(1, 2, 1)
    plt.plot(train_data, label="train", c=blue_color, linewidth=linewidth, zorder=zfar)
    skip_values = np.array(window_size*(np.nan,))
    flat_predict = np.ravel(inverse_y_train_predict)
    plot_predict = np.append(skip_values, flat_predict)
    plt.plot(plot_predict, label="train predict", c=orange_color, linewidth=linewidth, zorder=znear)
    plt.legend(loc='best')
    plt.title('Train, window '+str(window_size)+ ', model '+model_string)    
    
    plt.subplot(1, 2, 2)
    plt.plot(test_data, label="test", c=blue_color, linewidth=linewidth, zorder=zfar)
    skip_values = np.array(window_size*(np.nan,))
    flat_predict = np.ravel(inverse_y_test_predict)
    plot_predict = np.append(skip_values, flat_predict)
    plt.plot(plot_predict, label="test predict", c=orange_color, linewidth=linewidth, zorder=znear)
    plt.legend(loc='best')
    plt.title('Test, window '+str(window_size)+ ', model '+model_string) 
    
    plt.tight_layout()
    if filename is not None:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
sines_num_epochs = 50 # Early stopping will usually bring a halt to this a lot sooner
spots_num_epochs = 200
sines_training_length = 500
spots_training_length = 2332
sines_model_shapes = [[3], [3,3], [3,3,3,3,3], [5], [5,5], [5,5,5,5,5], [10], [10,10], [10,10,10,10,10], [25], [25,25], [25,25,25,25,25]]
spots_model_shapes = [[3], [3,3], [3,3,3], [4, 2, 1], [256, 128, 32]]
sines_windows = [5, 10, 20]
spots_windows = [5, 50, 100, 200] 
sines_linewidth = 2
spots_linewidth = 1

data_source = 'spots'

num_epochs = sines_num_epochs
training_length = sines_training_length
model_shapes = sines_model_shapes
windows = sines_windows
linewidth = sines_linewidth
if data_source != 'sines':
    num_epocsh = spots_num_epochs
    training_length = spots_training_length
    model_shapes = spots_model_shapes
    windows = spots_windows
    linewidth = spots_linewidth

for units_list in model_shapes:
    for window_size in windows:
        for model_type in ['RNN', 'Dense']:
            filename = data_source+'-'+model_type+'-units-'+str(units_list)+'-window-'+str(window_size)+'.png'
            build_and_show(model_type, data_source, units_list=units_list, window_size=window_size, 
                           training_length=training_length, epochs=num_epochs, 
                           linewidth=linewidth, filename=filename)